In [1]:
import csv
from pathlib import Path

In [2]:
scheduled_path = Path("raw-zip-scheduled")
scheduled_files = sorted(scheduled_path.glob("*.zip"))
print(len(scheduled_files))
scheduled_files[:5]

254


[PosixPath('raw-zip-scheduled/20050201SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20050301SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20050401SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20050501SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20050601SCLineOutages_csv.zip')]

In [3]:
import re
from collections import defaultdict, namedtuple

ScheduledOutage = namedtuple(
    "ScheduledOutage",
    [
        "timestamp",
        "ptid",
        "equipment_name",
        "scheduled_out_datetime",
        "scheduled_in_datetime",
        "bus1",
        "bus2",
        "voltage",
    ],
)
Interval = namedtuple("Interval", ["start", "end"])

equipment_name_pattern = r"^([A-Za-z0-9._ -]{8})-([A-Za-z0-9._ -]{8})_(\d{2,3})_(.+)$"

In [4]:
zip_path = scheduled_files[0]

In [5]:
import io
from datetime import datetime
from zipfile import ZipFile


def parse_scheduled_outage(row):
    equipment_name = row["Equipment Name"]
    m = re.match(equipment_name_pattern, equipment_name)
    if m is None:
        return None
    return ScheduledOutage(
        timestamp=datetime.strptime(row["Timestamp"], "%m/%d/%Y %H:%M:%S"),
        ptid=int(row["PTID"]),
        equipment_name=row["Equipment Name"],
        scheduled_out_datetime=datetime.strptime(
            row["Scheduled Out Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        scheduled_in_datetime=datetime.strptime(
            row["Scheduled In Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        bus1=m.group(1),
        bus2=m.group(2),
        voltage=int(m.group(3)),
    )


def list_csvs(zip_path):
    with ZipFile(zip_path) as z:
        return [i for i in sorted(z.namelist()) if i.endswith(".csv")]


def read_csv_from_zip(zip_path, csv_name):
    with ZipFile(zip_path) as z:
        with z.open(csv_name) as f:
            data = f.read()
        csv_reader = csv.DictReader(io.StringIO(data.decode("utf-8")))
        # Only keep rows with line outages
        data = [parse_scheduled_outage(row) for row in csv_reader]
        data = [row for row in data if row is not None]
    return data


def summarize_data(data):
    """Implement processing method described in this paper titled "Transmission grid outage statistics extracted from a webpage logging outages in northeast america" """
    # group by ptid
    by_ptid = defaultdict(list)
    for outage in data:
        by_ptid[outage.ptid].append(outage)
    # save (in, out) dates for each ptid
    summary = defaultdict(set)

    for ptid, rows in by_ptid.items():
        for row in rows:
            key = (
                row.scheduled_in_datetime,
                row.scheduled_out_datetime,
                row.equipment_name,
                row.bus1,
                row.bus2,
                row.voltage,
            )
            summary[ptid].add(key)
    return summary


# Example using an existing variable in the notebook:
zip_path = scheduled_files[0]
print(zip_path)
print("members:", list_csvs(zip_path))

# Read one file from the zip (text)
member = list_csvs(zip_path)[0]
data = read_csv_from_zip(zip_path, member)
print(len(data))
summary = summarize_data(data)
print(len(summary))

raw-zip-scheduled/20050201SCLineOutages_csv.zip
members: ['20050201SCLineOutages.csv', '20050202SCLineOutages.csv', '20050203SCLineOutages.csv', '20050204SCLineOutages.csv', '20050205SCLineOutages.csv', '20050206SCLineOutages.csv', '20050207SCLineOutages.csv', '20050208SCLineOutages.csv', '20050209SCLineOutages.csv', '20050210SCLineOutages.csv', '20050211SCLineOutages.csv', '20050212SCLineOutages.csv', '20050213SCLineOutages.csv', '20050214SCLineOutages.csv', '20050215SCLineOutages.csv', '20050216SCLineOutages.csv', '20050217SCLineOutages.csv', '20050218SCLineOutages.csv', '20050219SCLineOutages.csv', '20050220SCLineOutages.csv', '20050221SCLineOutages.csv', '20050222SCLineOutages.csv', '20050223SCLineOutages.csv', '20050224SCLineOutages.csv', '20050225SCLineOutages.csv', '20050226SCLineOutages.csv', '20050227SCLineOutages.csv', '20050228SCLineOutages.csv']
7215
37


In [6]:
from tqdm import tqdm

scheduled_outages = defaultdict(set)
for zip_path in tqdm(scheduled_files):
    for member in list_csvs(zip_path):
        data = read_csv_from_zip(zip_path, member)
        summary = summarize_data(data)
        for ptid, in_out_set in summary.items():
            scheduled_outages[ptid].update(in_out_set)

100%|██████████| 254/254 [23:20<00:00,  5.51s/it]


In [7]:
scheduled_outages.keys()

dict_keys([25094, 25190, 25228, 25239, 25243, 25255, 25553, 25559, 25560, 25565, 25566, 25569, 25876, 25894, 26002, 26004, 26040, 26053, 26058, 26187, 26193, 26264, 26477, 26478, 25138, 25563, 25217, 26112, 26186, 25517, 25143, 26010, 26011, 26175, 25231, 26106, 25291, 25550, 25084, 25505, 26685, 25222, 25681, 26274, 25012, 25014, 26500, 25251, 25182, 25326, 26433, 25564, 25024, 25164, 25220, 25857, 25147, 25223, 25253, 25279, 25906, 26042, 26055, 26057, 26153, 25122, 25123, 25329, 25330, 25304, 25124, 25265, 26508, 26014, 26015, 26056, 25300, 25152, 25303, 25536, 26022, 26023, 26145, 26070, 26077, 26090, 26109, 26498, 26108, 25066, 25241, 25585, 25519, 25245, 25145, 25308, 25097, 25095, 26285, 25554, 25186, 26144, 26065, 25283, 25117, 26696, 25229, 25769, 25770, 25514, 25494, 26239, 26080, 25061, 26176, 26183, 25882, 25116, 26043, 26059, 25895, 26113, 25037, 25154, 25212, 25052, 25829, 25830, 25104, 25105, 26653, 25582, 25202, 26161, 25902, 25581, 26173, 26174, 25360, 26232, 26191, 26

In [8]:
type(scheduled_outages[25013])

set

In [9]:
scheduled_outages[25013]

{(datetime.datetime(2013, 5, 21, 23, 59, 59),
  datetime.datetime(2013, 5, 20, 11, 19, 7),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2012, 6, 5, 16, 59, 59),
  datetime.datetime(2012, 6, 5, 10, 33, 24),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2017, 9, 4, 5, 59),
  datetime.datetime(2017, 9, 2, 5, 27),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2010, 9, 19, 7, 59, 59),
  datetime.datetime(2010, 9, 18, 17, 50, 21),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2018, 10, 20, 1, 59),
  datetime.datetime(2018, 10, 12, 8, 6),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2016, 10, 15, 23, 59),
  datetime.datetime(2016, 10, 14, 10, 24),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2018, 1, 25, 13, 59),
  datetime.datetime(2018, 1, 24, 12

In [10]:
import pandas as pd

actual_outage_csv_path = Path("processed-scheduled-outages.csv")

# build a flat table from the nested defaultdict
rows = []
for ptid, schedule_set in scheduled_outages.items():
    for info_tuple in sorted(schedule_set, key=lambda x: x[0]):
        rows.append(
            {
                "PTID": ptid,
                "Name": info_tuple[2],
                "Scheduled In Date/Time": info_tuple[0],
                "Scheduled Out Date/Time": info_tuple[1],
                # "Voltage": info_tuple[5],
                # "FirstBus": info_tuple[3],
                # "SecondBus": info_tuple[4],
            }
        )

rows = sorted(rows, key=lambda x: x["Scheduled In Date/Time"])
df = pd.DataFrame(rows)

# write out; use the existing json path with a .csv suffix
df.to_csv(actual_outage_csv_path, index=False)

print(f"saved {len(df)} rows to {actual_outage_csv_path}")

saved 12176653 rows to processed-scheduled-outages.csv
